# Trends, 2023–2025: Handling and Lateral-Transfer Injuries

The same rules and the same rate method applied to each year OSHA has published case detail: 2023, 2024, and 2025. Shows whether resident and patient handling injuries, and lateral-transfer injuries, are rising or falling, and gives a baseline for evaluating any intervention.

**Data** OSHA Injury Tracking Application case detail and Form 300A summaries, as published at https://www.osha.gov/itadata: 2023 (through 12-31-2023, OIICS release; summaries through 12-31-2024), 2024 (through 12-31-2025), 2025 (through 08-31-2026)
**Settings** Hospitals (NAICS 622), nursing care facilities (NAICS 6231)
**Source data** US Government work, 17 U.S.C. §105

### Comparing years

- 2023 was the first year establishments submitted case detail. Coverage can change between years as compliance changes; the number of establishments is reported for each year.
- The 2025 files were still open to submissions when downloaded (through 08-31-2026); 2023 and 2024 are closed.
- Rules are the frozen rules from `nursing-facility-handling/` and `handling-task-types/`. They were developed on 2025 narratives and validated on 2025 samples; their accuracy on earlier years is assumed, not measured.
- Rates use the population and hours filter from `injury-rates/`: 100+ employees, 500–4,000 hours per employee, case detail submitted.

## 1. Environment

In [1]:
!pip install -q duckdb pandas requests
import os, re, zipfile, requests, duckdb, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
FILES = {
 2023: ('https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2023_through_12-31-2023OIICS.zip',
        'https://www.osha.gov/sites/default/files/ITA_300A_Summary_Data_2023_through_12-31-2024.zip'),
 2024: ('https://www.osha.gov/sites/default/files/ITA_Case_Detail_Data_2024_through_12-31-2025.zip',
        'https://www.osha.gov/sites/default/files/ITA_300A_Summary_Data_2024_through_12-31-2025.zip'),
 2025: ('https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_08-31-2026.csv',
        'https://www.osha.gov/sites/default/largefiles/ITA_300A_Summary_Data_2025_through_08-31-2026.csv')}
def get_csv(url, stem):
    ext = url.rsplit('.', 1)[-1].lower(); path = f'data/{stem}.{ext}'
    if not os.path.exists(path):
        with requests.get(url, headers=HDRS, stream=True, timeout=1800) as r:
            r.raise_for_status()
            with open(path, 'wb') as f:
                for ch in r.iter_content(1 << 20): f.write(ch)
    if ext == 'zip':
        with zipfile.ZipFile(path) as z:
            name = max((i for i in z.infolist() if i.filename.lower().endswith('.csv')), key=lambda i: i.file_size).filename
            out = f'data/{stem}.csv'
            if not os.path.exists(out):
                with z.open(name) as src, open(out, 'wb') as dst: dst.write(src.read())
        os.remove(path); path = out
    print(f'{path}: {os.path.getsize(path)/1e6:.0f} MB'); return path

## 2. Frozen rules

In [2]:
TASKS = [
 ('lateral transfer between surfaces',
  r"\b(?:lateral(?:ly)? (?:transfer\w*|slid\w*|mov\w*)|bed to bed|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:operating|or|surgical|procedure|exam|examination|ct|cat scan|mri|x-?ray|imaging|radiology|cath\w*|ir|endoscopy|dialysis) (?:room )?(?:table|bed)|"
  r"(?:operating|or|surgical|procedure|exam|ct|mri|x-?ray|imaging) table to|"
  r"(?:bed|table|stretcher|gurney|cart|litter|cot) to (?:a |an |the |another |new |hospital |ems |shower |her |his )?(?:bed|table|stretcher|gurney|cart|litter|cot)|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:stretcher|gurney|litter|cot)\b|(?:stretcher|gurney|litter|cot) to|"
  r"slid\w* (?:the )?(?:resident|patient|pt|him|her) (?:over|across|onto|from))\b"),
 ('recovery from the floor',
  r"\b(?:off (?:of )?the (?:floor|ground)|from the (?:floor|ground)|up off|(?:found|was|lying) on the floor|lift\w* (?:\w+ ){0,2}(?:from|off) (?:the )?floor)\b"),
 ('catching or supporting a falling person',
  r"\b(?:catch\w*|caught (?:the |a )?(?:resident|patient|him|her)|began to fall|started to fall|was falling|"
  r"prevent\w* (?:\w+ ){0,2}from falling|keep \w+ from falling|lower\w* (?:\w+ ){0,2}to the (?:floor|ground)|(?:legs|knees) (?:gave out|buckled))\b"),
 ('seated or standing transfer',
  r"\b(?:(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair) to (?:a |the )?(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair)|"
  r"sit[- ]to[- ]stand|stand[- ]pivot|pivot\w*|into (?:the |a |her |his )?(?:wheel ?chair|chair|recliner|w/?c)|out of (?:the |a |her |his )?(?:wheel ?chair|chair|bed|recliner)|"
  r"on(?:to)? (?:the )?(?:toilet|commode)|off (?:the )?(?:toilet|commode)|(?:to|help\w* \w+) stand)\b"),
 ('repositioning in bed',
  r"\b(?:pull\w* (?:\w+ ){0,3}up (?:in|to)|boost\w*|reposition\w*|turn\w* (?:\w+ ){0,3}(?:in bed|over|on (?:her|his|their|the) side)|"
  r"roll\w* (?:the |a )?(?:resident|patient|pt|him|her)|draw ?sheet|turning sheet|up in bed|head of (?:the )?bed)\b"),
 ('ambulation', r"\b(?:ambulat\w*|walking (?:the |a |with (?:the |a )?)?(?:resident|patient|pt))\b"),
 ('transfer, type not stated', r"\btransfer\w*\b"),
]
SEATED_TO_STRETCHER = r"\b(?:wheel ?chair|chair|w/?c|recliner|geri ?chair) to (?:a |an |the )?(?:\w+ )?(?:stretcher|gurney|cot|table)\b"
def task(s):
    if re.search(SEATED_TO_STRETCHER, s): return 'seated or standing transfer'
    for name, p in TASKS:
        if re.search(p, s): return name
    return 'handling, task not stated'

PERSON   = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE   = r"patient handling|resident handling|resident care transfer"
SETTING = "CASE WHEN {n} LIKE '6231%' THEN 'nursing facility' WHEN {n} LIKE '622%' THEN 'hospital' END"

## 3. One year at a time

In [3]:
def run_year(year):
    cpath, spath = get_csv(FILES[year][0], f'cases_{year}'), get_csv(FILES[year][1], f'300a_{year}')
    con = duckdb.connect(':memory:')
    con.execute(f"CREATE TABLE c AS SELECT * FROM read_csv_auto('{cpath}', all_varchar=true, ignore_errors=true, sample_size=-1)")
    con.execute(f"CREATE TABLE s AS SELECT * FROM read_csv_auto('{spath}', all_varchar=true, ignore_errors=true, sample_size=-1)")
    def res(t):
        cols = {x.lower(): x for x in [r[0] for r in con.execute(f'DESCRIBE {t}').fetchall()]}
        return lambda *n: next(cols[k.lower()] for k in n if k.lower() in cols)
    cc, sc = res('c'), res('s')
    narr = [x for x in ['new_nar_before_incident', 'new_nar_what_happened', 'new_nar_injury_illness', 'new_nar_object_substance',
                        'nar_before_incident', 'nar_what_happened', 'nar_injury_illness', 'nar_object_substance']]
    have = {r[0].lower(): r[0] for r in con.execute('DESCRIBE c').fetchall()}
    new = [have[x] for x in narr[:4] if x in have]; old = [have[x] for x in narr[4:] if x in have]
    use = new if new else old
    assert use, f'{year}: no narrative columns found'
    print(year, 'narrative columns:', use)
    nar = "lower(concat_ws(' ', " + ', '.join(f'"{x}"' for x in use) + '))'
    cs = con.execute(f'''SELECT "{cc('establishment_id')}" AS establishment_id, {nar} AS narr,
           {SETTING.format(n='"' + cc('naics_code') + '"')} AS setting,
           CASE WHEN TRY_CAST("{cc('dafw_num_away')}" AS INT) >= 0 THEN TRY_CAST("{cc('dafw_num_away')}" AS INT) END AS dafw
         FROM c WHERE "{cc('naics_code')}" LIKE '6231%' OR "{cc('naics_code')}" LIKE '622%' ''').df()
    n = cs.narr.fillna('')
    cs['handling'] = (n.str.contains(PERSON, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)
    cs['lateral'] = False
    cs.loc[cs.handling, 'lateral'] = n[cs.handling].map(task).eq('lateral transfer between surfaces')
    cs['lost_time'] = cs.dafw.fillna(0) > 0
    est = con.execute(f'''SELECT "{sc('establishment_id')}" AS establishment_id,
           ANY_VALUE({SETTING.format(n='"' + sc('naics_code') + '"')}) AS setting,
           MAX(TRY_CAST("{sc('annual_average_employees')}" AS INT)) AS employees,
           MAX(TRY_CAST("{sc('total_hours_worked')}" AS BIGINT)) AS hours,
           MAX(COALESCE(TRY_CAST("{sc('total_deaths')}" AS INT), 0) + COALESCE(TRY_CAST("{sc('total_dafw_cases')}" AS INT), 0)
             + COALESCE(TRY_CAST("{sc('total_djtr_cases')}" AS INT), 0) + COALESCE(TRY_CAST("{sc('total_other_cases')}" AS INT), 0)) AS rec
         FROM s WHERE "{sc('naics_code')}" LIKE '6231%' OR "{sc('naics_code')}" LIKE '622%' GROUP BY 1''').df()
    con.close()
    est = est[(est.employees >= 100) & (est.hours > 0)].copy()
    est['hpe'] = est.hours / est.employees
    diag = est.groupby('setting').hpe.describe(percentiles=[.5, .95]).round(0).assign(year=year)
    est = est[est.hpe.between(500, 4000)]
    # A few establishments report hours per employee inside the plausible band but a total so large
    # that it cannot belong to one site. Drop the top 0.5% of hours within each setting and year.
    est = est[est.groupby('setting').hours.transform(lambda s: s <= s.quantile(0.995))]
    per = cs.groupby('establishment_id').agg(cases=('narr', 'size'), handling=('handling', 'sum'), lateral=('lateral', 'sum'))
    pop = est.merge(per, on='establishment_id', how='left').fillna({'cases': 0, 'handling': 0, 'lateral': 0})
    pop = pop[~((pop.rec > 0) & (pop.cases == 0))]
    counts = cs.groupby('setting').agg(cases=('narr', 'size'), handling=('handling', 'sum'), lateral=('lateral', 'sum'),
                                       handling_lost_time=('handling', lambda s: int((s & cs.loc[s.index, 'lost_time']).sum())),
                                       establishments_reporting=('establishment_id', 'nunique'))
    counts['handling_share'] = (counts.handling / counts.cases).round(3)
    r = pop.groupby('setting').agg(rate_establishments=('establishment_id', 'size'), hours=('hours', 'sum'),
                                   rate_cases=('cases', 'sum'), rate_handling=('handling', 'sum'), rate_lateral=('lateral', 'sum'))
    for k in ['cases', 'handling', 'lateral']: r[f'{k}_per_100_fte'] = (r[f'rate_{k}'] * 200000 / r.hours).round(3)
    out = counts.join(r).reset_index().assign(year=year).merge(diag[['50%', '95%', 'count']].rename(columns={'50%': 'hours_per_employee_median', '95%': 'hours_per_employee_p95', 'count': 'establishments_before_hours_filter'}), on='setting', how='left')
    for p in [cpath, spath]: os.remove(p)
    print(year, 'done'); return out
trend = pd.concat([run_year(y) for y in [2023, 2024, 2025]], ignore_index=True)
trend.to_csv('derived/trend-2023-2025.csv', index=False)

data/cases_2023.csv: 655 MB
data/300a_2023.csv: 87 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2023 narrative columns: ['NEW_NAR_BEFORE_INCIDENT', 'NEW_NAR_WHAT_HAPPENED', 'NEW_NAR_INJURY_ILLNESS', 'NEW_NAR_OBJECT_SUBSTANCE']
2023 done
data/cases_2024.csv: 520 MB
data/300a_2024.csv: 87 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2024 narrative columns: ['NEW_NAR_BEFORE_INCIDENT', 'NEW_NAR_WHAT_HAPPENED', 'NEW_NAR_INJURY_ILLNESS', 'NEW_NAR_OBJECT_SUBSTANCE']
2024 done
data/cases_2025.csv: 475 MB
data/300a_2025.csv: 88 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025 narrative columns: ['New_nar_before_incident', 'New_nar_what_happened', 'New_nar_injury_illness', 'New_nar_object_substance']
2025 done


## 4. Results

In [4]:
cols = ['year', 'establishments_reporting', 'cases', 'handling', 'handling_share', 'lateral', 'rate_establishments',
        'cases_per_100_fte', 'handling_per_100_fte', 'lateral_per_100_fte']
for s in ['hospital', 'nursing facility']:
    print(f'===== {s} ====='); print(trend[trend.setting == s][cols].to_string(index=False)); print()

===== hospital =====
 year  establishments_reporting  cases  handling  handling_share  lateral  rate_establishments  cases_per_100_fte  handling_per_100_fte  lateral_per_100_fte
 2023                      3917 168874     18909           0.112     1487                 3667              5.009                 0.572                0.045
 2024                      3701 138537     15172           0.110     1207                 3648              3.809                 0.424                0.035
 2025                      3967 181690     20685           0.114     1726                 3947              4.743                 0.544                0.046

===== nursing facility =====
 year  establishments_reporting  cases  handling  handling_share  lateral  rate_establishments  cases_per_100_fte  handling_per_100_fte  lateral_per_100_fte
 2023                      5354  42449      6380           0.150       29                 3579              6.651                 1.008                0.005
 2024  

### Hours reported per employee, by year

Rates depend on reported hours. This shows how plausible those hours are in each year, before the 500–4,000 filter.

In [5]:
trend[['year', 'setting', 'establishments_before_hours_filter', 'rate_establishments', 'hours_per_employee_median', 'hours_per_employee_p95', 'hours']].sort_values(['setting', 'year'])

,year,setting,establishments_before_hours_filter,rate_establishments,hours_per_employee_median,hours_per_employee_p95,hours
0,2023,hospital,4746.0,3667,1535.0,2293.0,6121377075
2,2024,hospital,4801.0,3648,1534.0,2224.0,6650126114
4,2025,hospital,4845.0,3947,1515.0,2233.0,7242393128
1,2023,nursing facility,5044.0,3579,1493.0,2258.0,933518533
3,2024,nursing facility,5095.0,3365,1502.0,2687.0,925046485
5,2025,nursing facility,5365.0,3592,1518.0,2405.0,977979303


  adding: derived/ (stored 0%)
  adding: derived/trend-2023-2025.csv (deflated 51%)


## Limitations

- Three years is a short series. Year-to-year changes can reflect changes in who submitted, not changes in risk.
- 2025 submissions were still open when the file was downloaded.
- Rule accuracy was measured on 2025 narratives only.
- Counts are as found by the rules, not adjusted for precision; adjustment would scale every year by the same factor and does not change the trend.